In [13]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/processed/loans_clean.csv")
df = df.rename(columns={"default": "is_default"})

df.insert(0, "loan_id", range(1, len(df) + 1))
df["applicant_id"] = df["loan_id"]

applicant_cols = ["applicant_id", "annual_inc", "income_bracket", "emp_length",
                  "emp_length_missing", "home_ownership", "addr_state", "dti",
                  "fico_avg", "delinq_2yrs", "pub_rec", "open_acc", "total_acc",
                  "revol_bal", "revol_util"]
loan_cols = ["loan_id", "applicant_id", "loan_amnt", "term", "int_rate",
             "installment", "grade", "purpose", "verification_status",
             "issue_year", "loan_to_income", "is_default"]

conn = sqlite3.connect("../data/processed/creditguard.db")
df[applicant_cols].to_sql("applicant_info", conn, if_exists="replace", index=False)
df[loan_cols].to_sql("loan_info", conn, if_exists="replace", index=False)
conn.execute("CREATE INDEX IF NOT EXISTS idx_loan_app ON loan_info(applicant_id)")
conn.commit()

print(pd.read_sql("SELECT COUNT(*) AS loans FROM loan_info", conn))

    loans
0  250000


In [7]:
def run(query, save_as=None):
    out = pd.read_sql(query, conn)
    if save_as:
        out.to_csv(f"../data/processed/{save_as}.csv", index=False)
    return out

In [8]:
run("""
SELECT COUNT(*) AS total_loans,
       ROUND(AVG(loan_amnt), 0) AS avg_loan_amount,
       ROUND(100.0 * SUM(is_default) / COUNT(*), 2) AS default_rate_pct
FROM loan_info;
""", "kpi_summary")

,total_loans,avg_loan_amount,default_rate_pct
0,250000,14423.0,19.88


In [9]:
run("""
SELECT a.income_bracket,
       COUNT(*) AS total_loans,
       SUM(l.is_default) AS defaults,
       ROUND(100.0 * SUM(l.is_default) / COUNT(*), 2) AS default_rate_pct
FROM loan_info l
JOIN applicant_info a ON l.applicant_id = a.applicant_id
GROUP BY a.income_bracket
ORDER BY default_rate_pct DESC;
""", "default_by_income")

,income_bracket,total_loans,defaults,default_rate_pct
0,<30k,18178,4392,24.16
1,30-60k,96280,21068,21.88
2,60-90k,73475,14274,19.43
3,90-120k,34419,5851,17.00
4,120k+,27648,4121,14.91


In [10]:
run("""
SELECT purpose,
       COUNT(*) AS total_loans,
       ROUND(100.0 * SUM(is_default) / COUNT(*), 2) AS default_rate_pct
FROM loan_info
GROUP BY purpose
ORDER BY default_rate_pct DESC;
""", "default_by_purpose")

,purpose,total_loans,default_rate_pct
0,small_business,2864,30.13
1,educational,53,26.42
2,house,1393,23.55
3,moving,1687,22.58
4,medical,2870,22.23
5,renewable_energy,183,21.86
6,other,14283,21.13
7,debt_consolidation,144961,20.97
8,vacation,1684,20.01
9,major_purchase,5557,18.86


In [11]:
run("""
SELECT CASE WHEN a.fico_avg < 680 THEN '1. Below 680'
            WHEN a.fico_avg < 700 THEN '2. 680-699'
            WHEN a.fico_avg < 720 THEN '3. 700-719'
            WHEN a.fico_avg < 740 THEN '4. 720-739'
            ELSE '5. 740+' END AS fico_band,
       COUNT(*) AS total_loans,
       ROUND(100.0 * SUM(l.is_default) / COUNT(*), 2) AS default_rate_pct
FROM loan_info l
JOIN applicant_info a ON l.applicant_id = a.applicant_id
GROUP BY fico_band
ORDER BY fico_band;
""", "default_by_fico")

,fico_band,total_loans,default_rate_pct
0,1. Below 680,85438,25.13
1,2. 680-699,67053,21.38
2,3. 700-719,45965,17.40
3,4. 720-739,24357,13.53
4,5. 740+,27187,9.56


In [12]:
run("""
SELECT a.addr_state AS state,
       COUNT(*) AS total_loans,
       ROUND(100.0 * SUM(l.is_default) / COUNT(*), 2) AS default_rate_pct
FROM loan_info l
JOIN applicant_info a ON l.applicant_id = a.applicant_id
GROUP BY a.addr_state
HAVING COUNT(*) >= 1000
ORDER BY default_rate_pct DESC;
""", "default_by_state")

,state,total_loans,default_rate_pct
0,MS,1187,29.91
1,OK,2278,23.84
2,AL,3167,23.27
3,AR,1846,23.13
4,MD,5748,22.89
5,LA,2930,22.73
6,NY,20420,22.09
7,MO,3807,21.72
8,FL,17785,21.52
9,NV,3795,21.50
